In [55]:
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

project_root = Path("../../..").resolve()
sys.path.append(str(project_root))

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_mean_pool

from utils import DataManager
from utils.plotter import actual_vs_prediction
torch.manual_seed(25)
np.random.seed(25)


In [43]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
df = DataManager('admet/mutagenicity', dataset_type='graph').dataset

train_dataset, test_dataset = train_test_split(df, test_size=0.2, random_state=25)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)    

In [45]:
class GINRegressor(nn.Module):
    def __init__(
        self,
        in_channels, 
        out_channels, 
        hidden_feat,
        num_layers,
    ):
        super().__init__()

        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()

        for _ in range(num_layers):
            mlp = nn.Sequential(
                nn.Linear(in_channels, 2 * hidden_feat),
                nn.BatchNorm1d(2 * hidden_feat),
                nn.ReLU(),
                nn.Linear(hidden_feat * 2, hidden_feat)

            )
            conv = GINConv(mlp, train_eps=True)

            self.convs.append(conv)
            self.norms.append(nn.BatchNorm1d(hidden_feat))
            in_channels = hidden_feat

        self.fc1 = nn.Linear(hidden_feat, hidden_feat // 2 )
        self.fc2 = nn.Linear(hidden_feat // 2, out_channels)
        self.norms1 = nn.BatchNorm1d(hidden_feat // 2)

    def _encode(self, x, edge_index):
        for convs, norms in zip(self.convs, self.norms):
            x = convs(x, edge_index)
            x = F.relu(norms(x))
        
        return x
    
    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch 

        h = self._encode(x, edge_index)
        h = global_mean_pool(h, batch)
        h = F.relu(self.norms1(self.fc1(h)))
        h = self.fc2(h)
        
        return h    

In [51]:
model = GINRegressor(in_channels=6, out_channels=2, hidden_feat=64, num_layers=2).to(device)
optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=15
)
criterion = nn.CrossEntropyLoss()

In [53]:
def train(model, optimizer, criterion, loader):
    model.train()
    total_loss = 0

    for data in loader:
        data = data.to(device)

        optimizer.zero_grad()

        # Model output: [num_graphs, num_classes]
        out = model(data)

        # Class labels: [num_graphs]
        y = data.y.view(-1).long()

        loss = criterion(out, y)
        loss.backward()

        # Prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0
        )

        optimizer.step()

        total_loss += loss.item() * data.num_graphs

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    y_true = []
    y_pred = []

    for data in loader:
        data = data.to(device)

        # Model output: [num_graphs, num_classes]
        out = model(data)

        # Get predicted class
        pred = out.argmax(dim=1)

        y_true.extend(
            data.y.view(-1).cpu().numpy()
        )

        y_pred.extend(
            pred.view(-1).cpu().numpy()
        )

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    return {
        "y_true": y_true,
        "y_pred": y_pred,

        "accuracy": accuracy_score(
            y_true,
            y_pred
        ),

        "precision": precision_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        ),

        "recall": recall_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        ),

        "f1": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        ),

        "confusion_matrix": confusion_matrix(
            y_true,
            y_pred
        )
    }

In [54]:

best_test_f1 = -np.inf
best_state = None

patience_counter = 0
early_stop_patience = 40


for epoch in range(500):

    # =========================
    # Training
    # =========================
    train_loss = train(
        model,
        optimizer,
        criterion,
        train_loader
    )

    # =========================
    # Evaluate
    # =========================
    test_metrics = evaluate(
        model,
        test_loader
    )

    # Scheduler monitors training loss
    scheduler.step(train_loss)

    # =========================
    # Save best model
    # =========================
    if test_metrics["f1"] > best_test_f1:

        best_test_f1 = test_metrics["f1"]

        best_state = {
            k: v.cpu().clone()
            for k, v in model.state_dict().items()
        }

        patience_counter = 0

    else:
        patience_counter += 1

    # =========================
    # Print progress
    # =========================
    if (epoch + 1) % 10 == 0:

        train_metrics = evaluate(
            model,
            train_loader
        )

        current_lr = optimizer.param_groups[0]["lr"]

        print(
            f"Epoch {epoch+1:03d} | "
            f"Loss {train_loss:.4f} | "
            f"Train Acc {train_metrics['accuracy']:.4f} | "
            f"Test Acc {test_metrics['accuracy']:.4f} | "
            f"Test F1 {test_metrics['f1']:.4f} | "
            f"LR {current_lr:.2e}"
        )

    # =========================
    # Early stopping
    # =========================
    if patience_counter >= early_stop_patience:

        print(
            f"Early stopping at epoch {epoch+1} "
            f"(best test F1={best_test_f1:.4f})"
        )

        break


# =========================
# Restore best model
# =========================

model.load_state_dict(best_state)


# =========================
# Final test evaluation
# =========================

final_metrics = evaluate(
    model,
    test_loader
)

print("\nFinal (best) test metrics:")
print(f"Accuracy : {final_metrics['accuracy']:.4f}")
print(f"Precision: {final_metrics['precision']:.4f}")
print(f"Recall   : {final_metrics['recall']:.4f}")
print(f"F1 Score : {final_metrics['f1']:.4f}")

print("\nConfusion Matrix:")
print(final_metrics["confusion_matrix"])

Epoch 010 | Loss 0.4878 | Train Acc 0.7754 | Test Acc 0.7736 | Test F1 0.7726 | LR 1.00e-03
Epoch 020 | Loss 0.4270 | Train Acc 0.8343 | Test Acc 0.7689 | Test F1 0.7696 | LR 1.00e-03
Epoch 030 | Loss 0.4040 | Train Acc 0.8550 | Test Acc 0.7689 | Test F1 0.7696 | LR 1.00e-03
Epoch 040 | Loss 0.3709 | Train Acc 0.8726 | Test Acc 0.7759 | Test F1 0.7749 | LR 1.00e-03
Epoch 050 | Loss 0.3244 | Train Acc 0.8603 | Test Acc 0.7807 | Test F1 0.7815 | LR 1.00e-03
Epoch 060 | Loss 0.2815 | Train Acc 0.9062 | Test Acc 0.7547 | Test F1 0.7556 | LR 1.00e-03
Early stopping at epoch 61 (best test F1=0.8013)

Final (best) test metrics:
Accuracy : 0.8019
Precision: 0.8014
Recall   : 0.8019
F1 Score : 0.8013

Confusion Matrix:
[[140  46]
 [ 38 200]]


In [58]:
print(classification_report(final_metrics['y_true'], final_metrics['y_pred']))

              precision    recall  f1-score   support

         0.0       0.79      0.75      0.77       186
         1.0       0.81      0.84      0.83       238

    accuracy                           0.80       424
   macro avg       0.80      0.80      0.80       424
weighted avg       0.80      0.80      0.80       424

